# NutriChat — RAG and LLM-only component ablations on Dev-200

This notebook evaluates the contribution of NutriChat's routing
and post-generation validation components on the 200-question
development benchmark.

It generates six new configurations.

## RAG safety-component ablations

1. Hybrid RRF + reranker, no router
2. Hybrid RRF + reranker, no validator
3. Hybrid RRF + reranker, no router or validator

## LLM-only safety-component ablations

4. LLM-only, no router
5. LLM-only, no validator
6. LLM-only, no router or validator

The notebook reuses fixed results from
`02a_compare_main_systems_dev200.ipynb` for:

- the full hybrid-reranked RAG system;
- hybrid RRF without reranking;
- dense retrieval with reranking; and
- LLM-only with the complete safety shell.

The existing `llm_only` result is therefore interpreted as:

`LLM-only + router + route-specific medical prompting + validator`

Dataset:

`data/eval_dataset_200_v2.json`

All RAG configurations use:

- candidate_k = 10
- final_k = 3
- retrieval-score threshold = None
- sentence_15_no_overlap chunking

A no-router configuration forces every query through the normal
question-answering path. It therefore also disables route-specific
medical prompt selection.

The notebook records API-call telemetry in a sidecar JSONL file.
The existing raw and judged CSV schemas are not modified.


In [1]:
from google.colab import drive, userdata

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/NutriChat-RAG/NutriChat"
%cd "$PROJECT_DIR"

!pip install -r requirements.txt
!pip install -e .


Mounted at /content/drive
/content/drive/MyDrive/NutriChat-RAG/NutriChat
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 86.5 MB/s eta 0:00:00
Obtaining file:///content/drive/MyDrive/NutriChat-RAG/NutriChat
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for nutrichat (pyproject.toml) ... done
  Created wheel for nutrichat: filename=nutrichat-0.1.0-0.editable-py3-none-any.whl size=2883 sha256=1f01a5b630821db0758e2d7b86f2a7a944bafc64c2ff2da5ac8b9541c5c5c6a4
  Stored in directory: /tmp/pip-ephem-wheel-cache-6o3t28ao/wheels/13/1f/4c/1f42e06f0c3ace164fb08724e739b140b59d7d29b5214e49d4
Successfully built nutrichat


In [2]:
from __future__ import annotations

import hashlib
import json
import random
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable, TypeVar

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from openai import (
    APIConnectionError,
    APITimeoutError,
    InternalServerError,
    OpenAI,
    RateLimitError,
)

from nutrichat.config import (
    ACTIVE_CHUNKING_STRATEGY,
    DEFAULT_MAX_NEW_TOKENS,
    DEFAULT_TEMPERATURE,
    EMBEDDING_MODEL,
    GENERATION_MODEL,
    JUDGE_MODEL,
    RERANKER_MODEL_NAME,
    ROUTER_MODEL,
    SystemSpec,
)
from nutrichat.data import load_eval_questions, load_index_artifact
from nutrichat.embeddings import load_embedding_model
from nutrichat.evaluation import run_system_evaluation_incremental
from nutrichat.generation import LLMOnlyPipeline, RAGPipeline
from nutrichat.judging import judge_dataframe_incremental
from nutrichat.reranking import load_reranker
from nutrichat.retrievers import (
    BM25Retriever,
    DenseRetriever,
    HybridRRFRetriever,
)


In [3]:
DEV_DATASET_PATH = Path(
    "data/eval_dataset_200_v2.json"
)

# Use a new version if any earlier component-ablation outputs were
# produced before telemetry was enabled. This avoids mixing logged
# and unlogged API calls.
RUN_ID = (
    "dev200_component_ablation_"
    "c10_f3_nogate_v2_telemetry"
)

RUN_ROOT = (
    Path("results")
    / "dev200_component_ablation"
    / RUN_ID
)

RAW_DIR = RUN_ROOT / "raw"
JUDGED_DIR = RUN_ROOT / "judged"
SUMMARY_DIR = RUN_ROOT / "summaries"
TELEMETRY_DIR = RUN_ROOT / "telemetry"

for directory in [
    RAW_DIR,
    JUDGED_DIR,
    SUMMARY_DIR,
    TELEMETRY_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

MAIN_COMPARISON_ROOT = (
    Path("results")
    / "dev200_main_system_comparison"
    / "dev200_main_systems_c10_f3_nogate_v1"
)

MAIN_JUDGED_DIR = (
    MAIN_COMPARISON_ROOT
    / "judged"
)

CANDIDATE_K = 10
FINAL_K = 3
RETRIEVAL_SCORE_THRESHOLD = None

DEBUG = False
DEBUG_QUESTION_COUNT = 3

USAGE_LOG_PATH = (
    TELEMETRY_DIR
    / "api_usage.jsonl"
)

# Keep False during incremental/resumed runs.
# Set True only for a completely new RUN_ID with no raw or judged files.
RESET_TELEMETRY = False

print("Component-ablation output:", RUN_ROOT)
print("Main comparison results:", MAIN_COMPARISON_ROOT)
print("API telemetry:", USAGE_LOG_PATH)


Component-ablation output: results/dev200_component_ablation/dev200_component_ablation_c10_f3_nogate_v2_telemetry
Main comparison results: results/dev200_main_system_comparison/dev200_main_systems_c10_f3_nogate_v1
API telemetry: results/dev200_component_ablation/dev200_component_ablation_c10_f3_nogate_v2_telemetry/telemetry/api_usage.jsonl


In [4]:
# ============================================================
# Load and validate the development benchmark
# ============================================================

dev_questions = load_eval_questions(str(DEV_DATASET_PATH))

assert len(dev_questions) == 200, (
    f"Expected 200 development questions, found {len(dev_questions)}."
)

question_ids = [str(item["id"]) for item in dev_questions]
assert len(set(question_ids)) == 200, "Development IDs are not unique."

answerable_count = sum(
    item["expected_behavior"] == "answer"
    for item in dev_questions
)

assert answerable_count == 140, (
    f"Expected 140 answerable development questions, found {answerable_count}."
)

questions_to_run = (
    dev_questions[:DEBUG_QUESTION_COUNT]
    if DEBUG
    else dev_questions
)

print("Questions selected:", len(questions_to_run))
print("Answerable questions in full dev set:", answerable_count)


Questions selected: 200
Answerable questions in full dev set: 140


In [5]:
# ============================================================
# Define the six new component-ablation systems
# ============================================================

RAG_COMPONENT_ABLATIONS = [
    SystemSpec(
        name=(
            "hybrid_reranker_no_router_"
            "c10_f3_nogate"
        ),
        retriever_name="hybrid_rrf",
        use_reranker=True,
        candidate_k=CANDIDATE_K,
        final_k=FINAL_K,
        min_retrieval_score=None,
        use_router=False,
        use_validator=True,
        experiment_group=(
            "dev200_rag_safety_ablation"
        ),
        ablation_factor="router",
        notes=(
            "Router disabled; all queries are forced "
            "through normal_nutrition_qa. Route-specific "
            "medical prompting is also disabled."
        ),
    ),
    SystemSpec(
        name=(
            "hybrid_reranker_no_validator_"
            "c10_f3_nogate"
        ),
        retriever_name="hybrid_rrf",
        use_reranker=True,
        candidate_k=CANDIDATE_K,
        final_k=FINAL_K,
        min_retrieval_score=None,
        use_router=True,
        use_validator=False,
        experiment_group=(
            "dev200_rag_safety_ablation"
        ),
        ablation_factor="validator",
        notes=(
            "Post-generation validator and fallback "
            "replacement disabled."
        ),
    ),
    SystemSpec(
        name=(
            "hybrid_reranker_no_router_no_validator_"
            "c10_f3_nogate"
        ),
        retriever_name="hybrid_rrf",
        use_reranker=True,
        candidate_k=CANDIDATE_K,
        final_k=FINAL_K,
        min_retrieval_score=None,
        use_router=False,
        use_validator=False,
        experiment_group=(
            "dev200_rag_safety_ablation"
        ),
        ablation_factor=(
            "router_and_validator"
        ),
        notes=(
            "Routing pathway and post-generation "
            "validator disabled. All queries use the "
            "normal RAG prompt, and generated answers "
            "are returned without validator fallback."
        ),
    ),
]


LLM_COMPONENT_ABLATIONS = [
    SystemSpec(
        name="llm_only_no_router",
        retriever_name="none",
        use_reranker=False,
        candidate_k=0,
        final_k=0,
        min_retrieval_score=None,
        use_router=False,
        use_validator=True,
        experiment_group=(
            "dev200_llm_safety_ablation"
        ),
        ablation_factor="llm_router",
        notes=(
            "Router and route-specific medical prompt "
            "selection disabled; post-generation "
            "validator remains enabled."
        ),
    ),
    SystemSpec(
        name="llm_only_no_validator",
        retriever_name="none",
        use_reranker=False,
        candidate_k=0,
        final_k=0,
        min_retrieval_score=None,
        use_router=True,
        use_validator=False,
        experiment_group=(
            "dev200_llm_safety_ablation"
        ),
        ablation_factor="llm_validator",
        notes=(
            "Router and route-specific medical prompt "
            "remain enabled; post-generation validator "
            "is disabled."
        ),
    ),
    SystemSpec(
        name=(
            "llm_only_no_router_no_validator"
        ),
        retriever_name="none",
        use_reranker=False,
        candidate_k=0,
        final_k=0,
        min_retrieval_score=None,
        use_router=False,
        use_validator=False,
        experiment_group=(
            "dev200_llm_safety_ablation"
        ),
        ablation_factor=(
            "llm_router_and_validator"
        ),
        notes=(
            "No external routing pathway, no "
            "route-specific medical prompt selection, "
            "and no post-generation validator."
        ),
    ),
]


assert len(RAG_COMPONENT_ABLATIONS) == 3
assert len(LLM_COMPONENT_ABLATIONS) == 3

expected_switches = {
    (False, True),
    (True, False),
    (False, False),
}

assert {
    (
        spec.use_router,
        spec.use_validator,
    )
    for spec in RAG_COMPONENT_ABLATIONS
} == expected_switches

assert {
    (
        spec.use_router,
        spec.use_validator,
    )
    for spec in LLM_COMPONENT_ABLATIONS
} == expected_switches


print("RAG component ablations")
print("-" * 70)

for spec in RAG_COMPONENT_ABLATIONS:
    print(
        spec.name,
        "| router =", spec.use_router,
        "| validator =", spec.use_validator,
    )


print("\nLLM-only component ablations")
print("-" * 70)

for spec in LLM_COMPONENT_ABLATIONS:
    print(
        spec.name,
        "| router =", spec.use_router,
        "| validator =", spec.use_validator,
    )


RAG component ablations
----------------------------------------------------------------------
hybrid_reranker_no_router_c10_f3_nogate | router = False | validator = True
hybrid_reranker_no_validator_c10_f3_nogate | router = True | validator = False
hybrid_reranker_no_router_no_validator_c10_f3_nogate | router = False | validator = False

LLM-only component ablations
----------------------------------------------------------------------
llm_only_no_router | router = False | validator = True
llm_only_no_validator | router = True | validator = False
llm_only_no_router_no_validator | router = False | validator = False


In [7]:
# ============================================================
# Load local retrieval and reranking models
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

from nutrichat.telemetry import (
    ACTIVE_SYSTEM,
    UsageLoggingClient,
)


if RESET_TELEMETRY:
    existing_result_files = (
        list(RAW_DIR.glob("*.csv"))
        + list(JUDGED_DIR.glob("*.csv"))
    )

    if existing_result_files:
        raise RuntimeError(
            "Telemetry cannot be reset while raw or judged "
            "result files exist for this RUN_ID. Use a new "
            "RUN_ID instead."
        )

    if USAGE_LOG_PATH.exists():
        USAGE_LOG_PATH.unlink()


base_client = OpenAI(
    base_url=(
        "https://integrate.api.nvidia.com/v1"
    ),
    api_key=userdata.get(
        "NVIDIA_API_KEY"
    ),
    timeout=120.0,
    max_retries=3,
)

client = UsageLoggingClient(
    wrapped=base_client,
    log_path=USAGE_LOG_PATH,
)

index_dir = Path(
    "artifacts"
) / f"index_{ACTIVE_CHUNKING_STRATEGY}"

chunks, embeddings_np = load_index_artifact(index_dir)

embeddings = torch.as_tensor(
    embeddings_np,
    dtype=torch.float32,
    device=DEVICE,
)

embedding_model = load_embedding_model(
    EMBEDDING_MODEL,
    device=DEVICE,
)

reranker_model = load_reranker(
    RERANKER_MODEL_NAME,
    device=DEVICE,
)

assert (
    embeddings.shape[1]
    == embedding_model.get_sentence_embedding_dimension()
), (
    f"Embedding mismatch: saved embeddings are {embeddings.shape[1]}D, "
    f"but {EMBEDDING_MODEL} produces "
    f"{embedding_model.get_sentence_embedding_dimension()}D."
)

print("Chunks:", len(chunks))
print("Embedding shape:", tuple(embeddings.shape))
print("Embedding model:", EMBEDDING_MODEL)
print("Reranker:", RERANKER_MODEL_NAME)
print("Generator:", GENERATION_MODEL)
print("Router/validator:", ROUTER_MODEL)
print("Judge:", JUDGE_MODEL)
print("API telemetry:", USAGE_LOG_PATH)


Device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

Chunks: 1212
Embedding shape: (1212, 384)
Embedding model: BAAI/bge-small-en-v1.5
Reranker: BAAI/bge-reranker-base
Generator: nvidia/llama-3.3-nemotron-super-49b-v1
Router/validator: openai/gpt-oss-120b
Judge: openai/gpt-oss-120b
API telemetry: results/dev200_component_ablation/dev200_component_ablation_c10_f3_nogate_v2_telemetry/telemetry/api_usage.jsonl


/tmp/ipykernel_1171/3626024095.py:71: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  == embedding_model.get_sentence_embedding_dimension()


In [8]:
# ============================================================
# Build retrievers and pipelines
# ============================================================

dense_retriever = DenseRetriever(
    chunks=chunks,
    embeddings=embeddings,
    embedding_model=embedding_model,
)

bm25_retriever = BM25Retriever(
    chunks=chunks,
)

hybrid_retriever = HybridRRFRetriever(
    dense_retriever=dense_retriever,
    bm25_retriever=bm25_retriever,
)

retrievers = {
    "dense": dense_retriever,
    "bm25": bm25_retriever,
    "hybrid_rrf": hybrid_retriever,
}

rag_pipeline = RAGPipeline(
    client=client,
    generation_model=GENERATION_MODEL,
    retrievers=retrievers,
    reranker_model=reranker_model,
)

llm_pipeline = LLMOnlyPipeline(
    client=client,
    generation_model=GENERATION_MODEL,
)


## Verify component-ablation support

The project code must expose `use_router` and `use_validator`
for both `RAGPipeline.answer()` and `LLMOnlyPipeline.answer()`.

The non-RAG branch of `run_system_evaluation_incremental()` must
also forward these values from `SystemSpec` to the LLM-only
pipeline.


In [9]:
import inspect


required_answer_parameters = {
    "system_variant",
    "use_router",
    "use_validator",
}


rag_answer_parameters = set(
    inspect.signature(
        rag_pipeline.answer
    ).parameters
)

llm_answer_parameters = set(
    inspect.signature(
        llm_pipeline.answer
    ).parameters
)


missing_rag_parameters = (
    required_answer_parameters
    - rag_answer_parameters
)

missing_llm_parameters = (
    required_answer_parameters
    - llm_answer_parameters
)


if missing_rag_parameters:
    raise RuntimeError(
        "RAGPipeline.answer() is missing: "
        f"{sorted(missing_rag_parameters)}"
    )

if missing_llm_parameters:
    raise RuntimeError(
        "LLMOnlyPipeline.answer() is missing: "
        f"{sorted(missing_llm_parameters)}. "
        "Apply the LLM-only router/validator changes "
        "to nutrichat/generation.py before running "
        "this notebook."
    )


for spec in (
    RAG_COMPONENT_ABLATIONS
    + LLM_COMPONENT_ABLATIONS
):
    assert hasattr(spec, "use_router")
    assert hasattr(spec, "use_validator")


print(
    "Component-ablation interface check passed."
)


Component-ablation interface check passed.


In [10]:
# ============================================================
# Retry helper for remote API failures and rate limits
# ============================================================

T = TypeVar("T")


def run_with_api_backoff(
    operation_name: str,
    operation: Callable[[], T],
    max_retries: int = 12,
) -> T:
    attempt = 0

    while True:
        try:
            return operation()

        except RateLimitError:
            attempt += 1
            if attempt > max_retries:
                raise

            wait_seconds = min(
                900.0,
                60.0 * (2 ** min(attempt - 1, 4)),
            ) + random.uniform(2.0, 12.0)

            print(
                f"{operation_name}: HTTP 429. "
                f"Waiting {wait_seconds:.1f} seconds."
            )
            time.sleep(wait_seconds)

        except (
            APITimeoutError,
            APIConnectionError,
            InternalServerError,
        ) as error:
            attempt += 1
            if attempt > max_retries:
                raise

            wait_seconds = min(
                300.0,
                20.0 * (2 ** min(attempt - 1, 3)),
            ) + random.uniform(1.0, 8.0)

            print(
                f"{operation_name}: {type(error).__name__}. "
                f"Waiting {wait_seconds:.1f} seconds."
            )
            time.sleep(wait_seconds)


## Generate the six new component ablations

The first loop runs the three hybrid-reranked RAG variants.
The second loop runs the three LLM-only variants.

Each system writes to a separate incremental CSV. The active
system name is attached to every API telemetry record.

If Colab disconnects, reconnect and rerun this cell. Completed
question IDs will be skipped.


In [11]:
def run_component_ablation_group(
    system_specs,
    pipeline,
    is_rag_system: bool,
) -> None:
    for spec in system_specs:
        suffix = "_debug" if DEBUG else ""

        raw_path = (
            RAW_DIR
            / f"{spec.name}{suffix}.csv"
        )

        print("\n" + "=" * 88)
        print("Running:", spec.name)
        print(
            "Pipeline:",
            "RAG"
            if is_rag_system
            else "LLM-only",
        )
        print(
            "Router:",
            spec.use_router,
            "| Validator:",
            spec.use_validator,
        )
        print("=" * 88)

        active_system_token = (
            ACTIVE_SYSTEM.set(
                spec.name
            )
        )

        try:
            raw_df = run_with_api_backoff(
                operation_name=(
                    f"Raw evaluation: {spec.name}"
                ),
                operation=(
                    lambda spec=spec,
                    raw_path=raw_path:
                    run_system_evaluation_incremental(
                        eval_questions=(
                            questions_to_run
                        ),
                        system_spec=spec,
                        pipeline=pipeline,
                        output_path=raw_path,
                        temperature=(
                            spec.temperature
                        ),
                        max_new_tokens=(
                            spec.max_new_tokens
                        ),
                        is_rag_system=(
                            is_rag_system
                        ),
                    )
                ),
            )

        finally:
            ACTIVE_SYSTEM.reset(
                active_system_token
            )

        expected_rows = len(
            questions_to_run
        )

        assert len(raw_df) == expected_rows

        assert (
            raw_df["id"]
            .astype(str)
            .nunique()
            == expected_rows
        )

        assert (
            raw_df["system"]
            .astype(str)
            .eq(spec.name)
            .all()
        ), (
            f"{spec.name}: the output system name "
            "does not match the SystemSpec name. "
            "Confirm that evaluation.py forwards "
            "system_variant for the LLM-only branch."
        )

        print(
            spec.name,
            raw_df.shape,
        )


run_component_ablation_group(
    system_specs=RAG_COMPONENT_ABLATIONS,
    pipeline=rag_pipeline,
    is_rag_system=True,
)

run_component_ablation_group(
    system_specs=LLM_COMPONENT_ABLATIONS,
    pipeline=llm_pipeline,
    is_rag_system=False,
)



Running: hybrid_reranker_no_router_c10_f3_nogate
Pipeline: RAG
Router: False | Validator: True
Found existing file with 200 completed questions.
Skipping already completed question: A001
Skipping already completed question: A002
Skipping already completed question: A003
Skipping already completed question: A004
Skipping already completed question: A005
Skipping already completed question: A006
Skipping already completed question: A007
Skipping already completed question: A008
Skipping already completed question: A009
Skipping already completed question: A010
Skipping already completed question: A011
Skipping already completed question: A012
Skipping already completed question: A013
Skipping already completed question: A014
Skipping already completed question: A015
Skipping already completed question: A016
Skipping already completed question: A017
Skipping already completed question: A018
Skipping already completed question: A019
Skipping already completed question: A020
Skipping alrea

## Judge every completed system output

Judging is also incremental. Existing valid `(system, id)` judgments are preserved and skipped.


In [16]:
raw_pattern = "*_debug.csv" if DEBUG else "*.csv"
raw_paths = sorted(RAW_DIR.glob(raw_pattern))

assert raw_paths, f"No raw files were found in {RAW_DIR}."

for raw_path in raw_paths:
    raw_df = pd.read_csv(raw_path)

    expected_rows = len(questions_to_run)

    assert len(raw_df) == expected_rows, (
        f"{raw_path.name}: expected {expected_rows} rows, "
        f"found {len(raw_df)}."
    )
    assert (
        raw_df["id"]
        .astype(str)
        .nunique()
        == expected_rows
    )

    judged_path = (
        JUDGED_DIR
        / f"{raw_path.stem}_judged.csv"
    )

    print("\n" + "=" * 88)
    print("Judging:", raw_path.stem)
    print("=" * 88)

    active_system_token = ACTIVE_SYSTEM.set(
        raw_path.stem
    )

    try:
        judged_df = run_with_api_backoff(
            operation_name=(
                f"Judging: {raw_path.stem}"
            ),
            operation=(
                lambda raw_df=raw_df,
                judged_path=judged_path:
                judge_dataframe_incremental(
                    eval_df=raw_df,
                    client=client,
                    judge_model=JUDGE_MODEL,
                    output_path=judged_path,
                )
            ),
        )

    finally:
        ACTIVE_SYSTEM.reset(
            active_system_token
        )

    assert len(judged_df) == expected_rows, (
        f"{judged_path.name}: expected {expected_rows} judged rows, "
        f"found {len(judged_df)}."
    )
    assert (
        judged_df["id"]
        .astype(str)
        .nunique()
        == expected_rows
    )
    assert judged_df["pass"].notna().all()

    print(raw_path.stem, judged_df.shape)



Judging: hybrid_reranker_no_router_c10_f3_nogate
Found existing judged file with 200 completed rows.
Skipping already judged row: hybrid_reranker_no_router_c10_f3_nogate::A001
Skipping already judged row: hybrid_reranker_no_router_c10_f3_nogate::A002
Skipping already judged row: hybrid_reranker_no_router_c10_f3_nogate::A003
Skipping already judged row: hybrid_reranker_no_router_c10_f3_nogate::A004
Skipping already judged row: hybrid_reranker_no_router_c10_f3_nogate::A005
Skipping already judged row: hybrid_reranker_no_router_c10_f3_nogate::A006
Skipping already judged row: hybrid_reranker_no_router_c10_f3_nogate::A007
Skipping already judged row: hybrid_reranker_no_router_c10_f3_nogate::A008
Skipping already judged row: hybrid_reranker_no_router_c10_f3_nogate::A009
Skipping already judged row: hybrid_reranker_no_router_c10_f3_nogate::A010
Skipping already judged row: hybrid_reranker_no_router_c10_f3_nogate::A011
Skipping already judged row: hybrid_reranker_no_router_c10_f3_nogate::A01

## Summarize API calls, tokens, and cost

The JSONL telemetry file records actual API attempts made during
this notebook, including failed attempts and retries.

The deployment path includes:

- router calls;
- generator calls; and
- validator calls.

Judge calls are evaluation overhead and are reported separately.

NVIDIA's hosted developer endpoint does not provide a universal
public per-token price for these models. The notebook therefore
always reports calls and tokens, but calculates a dollar estimate
only after applicable per-million-token prices are entered below.


In [17]:
# ============================================================
# Pricing assumptions
# ============================================================

# Enter only prices that actually apply to the endpoint/account
# used for this run. Leave values as None when no applicable
# public or invoiced per-token price exists.
#
# Units: USD per 1,000,000 tokens.

MODEL_PRICING_USD_PER_MILLION = {}

for model_name in {
    GENERATION_MODEL,
    ROUTER_MODEL,
    JUDGE_MODEL,
}:
    MODEL_PRICING_USD_PER_MILLION[
        model_name
    ] = {
        "input": None,
        "output": None,
    }


# Example of the required format for a priced endpoint:
#
# MODEL_PRICING_USD_PER_MILLION[
#     "provider/model-name"
# ] = {
#     "input": 1.00,
#     "output": 3.00,
# }


PRICING_CURRENCY = "USD"

PRICING_SOURCE = (
    "No applicable per-token price entered for the "
    "NVIDIA hosted developer endpoint."
)

PRICING_DATE = None

# Set to 0.0 only when your NVIDIA account/dashboard confirms
# that this experimental run produced no monetary charge.
# Otherwise leave as None.
CONFIRMED_OUT_OF_POCKET_COST_USD = None


pricing_assumptions = {
    "currency": PRICING_CURRENCY,
    "pricing_date": PRICING_DATE,
    "pricing_source": PRICING_SOURCE,
    "models": (
        MODEL_PRICING_USD_PER_MILLION
    ),
    "confirmed_out_of_pocket_cost_usd": (
        CONFIRMED_OUT_OF_POCKET_COST_USD
    ),
}

pricing_path = (
    TELEMETRY_DIR
    / "pricing_assumptions.json"
)

pricing_path.write_text(
    json.dumps(
        pricing_assumptions,
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)

print("Pricing assumptions:", pricing_path)


Pricing assumptions: results/dev200_component_ablation/dev200_component_ablation_c10_f3_nogate_v2_telemetry/telemetry/pricing_assumptions.json


In [18]:
# ============================================================
# Read and validate API telemetry
# ============================================================

if not USAGE_LOG_PATH.exists():
    raise FileNotFoundError(
        f"Telemetry log not found: {USAGE_LOG_PATH}"
    )

api_usage = pd.read_json(
    USAGE_LOG_PATH,
    lines=True,
)

required_telemetry_columns = {
    "timestamp_utc",
    "system",
    "call_type",
    "model",
    "success",
    "latency_seconds",
    "input_tokens",
    "output_tokens",
    "total_tokens",
    "error_type",
}

missing_columns = (
    required_telemetry_columns
    - set(api_usage.columns)
)

if missing_columns:
    raise ValueError(
        "Telemetry is missing columns: "
        f"{sorted(missing_columns)}"
    )


for column in [
    "input_tokens",
    "output_tokens",
    "total_tokens",
    "latency_seconds",
]:
    api_usage[column] = pd.to_numeric(
        api_usage[column],
        errors="coerce",
    )


unknown_system_rows = api_usage[
    api_usage["system"].fillna("unknown")
    == "unknown"
]

if not unknown_system_rows.empty:
    raise ValueError(
        f"Found {len(unknown_system_rows)} telemetry "
        "records with system='unknown'. Start a new "
        "RUN_ID and rerun after applying the "
        "ACTIVE_SYSTEM changes."
    )


successful_usage = api_usage[
    api_usage["success"] == True  # noqa: E712
].copy()

failed_usage = api_usage[
    api_usage["success"] != True  # noqa: E712
].copy()


def price_value(
    model_name: str,
    token_type: str,
):
    model_pricing = (
        MODEL_PRICING_USD_PER_MILLION
        .get(
            str(model_name),
            {},
        )
    )

    value = model_pricing.get(
        token_type
    )

    if value is None:
        return np.nan

    return float(value)


successful_usage[
    "input_price_usd_per_million"
] = successful_usage["model"].map(
    lambda model: price_value(
        model,
        "input",
    )
)

successful_usage[
    "output_price_usd_per_million"
] = successful_usage["model"].map(
    lambda model: price_value(
        model,
        "output",
    )
)

successful_usage[
    "price_available"
] = (
    successful_usage[
        "input_price_usd_per_million"
    ].notna()
    &
    successful_usage[
        "output_price_usd_per_million"
    ].notna()
    &
    successful_usage[
        "input_tokens"
    ].notna()
    &
    successful_usage[
        "output_tokens"
    ].notna()
)

successful_usage[
    "estimated_cost_usd"
] = np.where(
    successful_usage["price_available"],
    (
        successful_usage["input_tokens"]
        / 1_000_000
        * successful_usage[
            "input_price_usd_per_million"
        ]
        +
        successful_usage["output_tokens"]
        / 1_000_000
        * successful_usage[
            "output_price_usd_per_million"
        ]
    ),
    np.nan,
)


def sum_with_nan(values: pd.Series):
    return values.sum(
        min_count=1
    )


def p90(values: pd.Series):
    return values.quantile(0.90)


usage_by_call_type = (
    successful_usage
    .groupby(
        [
            "system",
            "call_type",
            "model",
        ],
        dropna=False,
    )
    .agg(
        successful_calls=(
            "call_type",
            "size",
        ),
        input_tokens=(
            "input_tokens",
            sum_with_nan,
        ),
        output_tokens=(
            "output_tokens",
            sum_with_nan,
        ),
        total_tokens=(
            "total_tokens",
            sum_with_nan,
        ),
        calls_with_token_usage=(
            "total_tokens",
            "count",
        ),
        priced_calls=(
            "price_available",
            "sum",
        ),
        estimated_cost_usd=(
            "estimated_cost_usd",
            sum_with_nan,
        ),
        median_call_latency_seconds=(
            "latency_seconds",
            "median",
        ),
        p90_call_latency_seconds=(
            "latency_seconds",
            p90,
        ),
    )
    .reset_index()
)

usage_by_call_type[
    "pricing_complete"
] = (
    usage_by_call_type[
        "priced_calls"
    ]
    ==
    usage_by_call_type[
        "successful_calls"
    ]
)

usage_by_call_type.loc[
    ~usage_by_call_type[
        "pricing_complete"
    ],
    "estimated_cost_usd",
] = np.nan


failed_by_call_type = (
    failed_usage
    .groupby(
        [
            "system",
            "call_type",
            "model",
            "error_type",
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="failed_attempts"
    )
)


DEPLOYMENT_CALL_TYPES = {
    "router",
    "generator",
    "validator",
}

deployment_usage = successful_usage[
    successful_usage["call_type"].isin(
        DEPLOYMENT_CALL_TYPES
    )
].copy()

judge_usage = successful_usage[
    successful_usage["call_type"]
    == "judge"
].copy()


def summarize_scope(
    dataframe: pd.DataFrame,
    scope_name: str,
) -> pd.DataFrame:
    if dataframe.empty:
        return pd.DataFrame(
            columns=[
                "system",
                "scope",
                "successful_calls",
                "input_tokens",
                "output_tokens",
                "total_tokens",
                "calls_with_token_usage",
                "priced_calls",
                "estimated_cost_usd",
                "pricing_complete",
            ]
        )

    summary = (
        dataframe
        .groupby(
            "system",
            dropna=False,
        )
        .agg(
            successful_calls=(
                "call_type",
                "size",
            ),
            input_tokens=(
                "input_tokens",
                sum_with_nan,
            ),
            output_tokens=(
                "output_tokens",
                sum_with_nan,
            ),
            total_tokens=(
                "total_tokens",
                sum_with_nan,
            ),
            calls_with_token_usage=(
                "total_tokens",
                "count",
            ),
            priced_calls=(
                "price_available",
                "sum",
            ),
            estimated_cost_usd=(
                "estimated_cost_usd",
                sum_with_nan,
            ),
        )
        .reset_index()
    )

    summary["scope"] = scope_name

    summary["pricing_complete"] = (
        summary["priced_calls"]
        ==
        summary["successful_calls"]
    )

    summary.loc[
        ~summary["pricing_complete"],
        "estimated_cost_usd",
    ] = np.nan

    return summary


deployment_summary = summarize_scope(
    deployment_usage,
    "deployment",
)

judge_summary = summarize_scope(
    judge_usage,
    "evaluation_judge",
)

cost_summary = pd.concat(
    [
        deployment_summary,
        judge_summary,
    ],
    ignore_index=True,
)


raw_question_counts = []

for raw_path in sorted(
    RAW_DIR.glob(
        "*_debug.csv"
        if DEBUG
        else "*.csv"
    )
):
    raw_dataframe = pd.read_csv(
        raw_path
    )

    raw_question_counts.append(
        {
            "system": raw_path.stem,
            "n_questions": int(
                raw_dataframe["id"]
                .astype(str)
                .nunique()
            ),
        }
    )

question_count_table = pd.DataFrame(
    raw_question_counts
)

cost_summary = cost_summary.merge(
    question_count_table,
    on="system",
    how="left",
)

cost_summary[
    "total_tokens_per_question"
] = (
    cost_summary["total_tokens"]
    / cost_summary["n_questions"]
)

cost_summary[
    "estimated_cost_usd_per_question"
] = (
    cost_summary["estimated_cost_usd"]
    / cost_summary["n_questions"]
)


usage_by_call_type_path = (
    TELEMETRY_DIR
    / "api_usage_by_call_type.csv"
)

failed_calls_path = (
    TELEMETRY_DIR
    / "api_failed_attempts.csv"
)

cost_summary_path = (
    TELEMETRY_DIR
    / "api_cost_summary.csv"
)

api_usage.to_csv(
    TELEMETRY_DIR
    / "api_usage_all_attempts.csv",
    index=False,
)

usage_by_call_type.to_csv(
    usage_by_call_type_path,
    index=False,
)

failed_by_call_type.to_csv(
    failed_calls_path,
    index=False,
)

cost_summary.to_csv(
    cost_summary_path,
    index=False,
)


display(
    usage_by_call_type.style
    .format(
        {
            "input_tokens": "{:,.0f}",
            "output_tokens": "{:,.0f}",
            "total_tokens": "{:,.0f}",
            "estimated_cost_usd": "${:,.6f}",
            "median_call_latency_seconds": "{:.2f}",
            "p90_call_latency_seconds": "{:.2f}",
        },
        na_rep="--",
    )
    .hide(axis="index")
    .set_caption(
        "API usage by system and call type"
    )
)

display(
    cost_summary.style
    .format(
        {
            "input_tokens": "{:,.0f}",
            "output_tokens": "{:,.0f}",
            "total_tokens": "{:,.0f}",
            "total_tokens_per_question": "{:,.1f}",
            "estimated_cost_usd": "${:,.6f}",
            "estimated_cost_usd_per_question": "${:,.6f}",
        },
        na_rep="--",
    )
    .hide(axis="index")
    .set_caption(
        "Deployment and evaluation API-cost summary"
    )
)


missing_token_usage = int(
    successful_usage[
        "total_tokens"
    ].isna().sum()
)

print("Successful API calls:", len(successful_usage))
print("Failed API attempts:", len(failed_usage))
print(
    "Successful calls without returned token usage:",
    missing_token_usage,
)

if cost_summary[
    "estimated_cost_usd"
].notna().all():
    print(
        "Token-based USD estimate calculated "
        "from the entered prices."
    )
else:
    print(
        "Dollar estimate is intentionally blank because "
        "one or more applicable per-token prices or token "
        "usage values are unavailable."
    )

if (
    CONFIRMED_OUT_OF_POCKET_COST_USD
    is not None
):
    print(
        "Confirmed out-of-pocket charge:",
        f"${CONFIRMED_OUT_OF_POCKET_COST_USD:,.2f}",
    )

print("Saved:", usage_by_call_type_path)
print("Saved:", failed_calls_path)
print("Saved:", cost_summary_path)


system,call_type,model,successful_calls,input_tokens,output_tokens,total_tokens,calls_with_token_usage,priced_calls,estimated_cost_usd,median_call_latency_seconds,p90_call_latency_seconds,pricing_complete
hybrid_reranker_no_router_c10_f3_nogate,generator,nvidia/llama-3.3-nemotron-super-49b-v1,200,"302,265","39,478","341,743",200,0,--,13.50,32.11,False
hybrid_reranker_no_router_c10_f3_nogate,judge,openai/gpt-oss-120b,201,"495,045","88,565","583,610",201,0,--,3.87,6.28,False
hybrid_reranker_no_router_c10_f3_nogate,validator,openai/gpt-oss-120b,198,"161,581","29,970","191,551",198,0,--,1.04,1.43,False
hybrid_reranker_no_router_no_validator_c10_f3_nogate,generator,nvidia/llama-3.3-nemotron-super-49b-v1,200,"302,265","39,932","342,197",200,0,--,13.68,31.52,False
hybrid_reranker_no_router_no_validator_c10_f3_nogate,judge,openai/gpt-oss-120b,200,"493,833","86,766","580,599",200,0,--,3.10,4.54,False
hybrid_reranker_no_validator_c10_f3_nogate,generator,nvidia/llama-3.3-nemotron-super-49b-v1,185,"287,937","36,337","324,274",185,0,--,12.98,36.29,False
hybrid_reranker_no_validator_c10_f3_nogate,judge,openai/gpt-oss-120b,200,"478,964","83,493","562,457",200,0,--,2.72,4.14,False
hybrid_reranker_no_validator_c10_f3_nogate,router,openai/gpt-oss-120b,197,"98,983","28,294","127,277",197,0,--,1.00,1.57,False
llm_only_no_router,generator,nvidia/llama-3.3-nemotron-super-49b-v1,200,"29,127","91,589","120,716",200,0,--,31.42,57.28,False
llm_only_no_router,judge,openai/gpt-oss-120b,200,"353,303","75,625","428,928",200,0,--,2.59,3.82,False


system,successful_calls,input_tokens,output_tokens,total_tokens,calls_with_token_usage,priced_calls,estimated_cost_usd,scope,pricing_complete,n_questions,total_tokens_per_question,estimated_cost_usd_per_question
hybrid_reranker_no_router_c10_f3_nogate,398,"463,846","69,448","533,294",398,0,--,deployment,False,200,"2,666.5",--
hybrid_reranker_no_router_no_validator_c10_f3_nogate,200,"302,265","39,932","342,197",200,0,--,deployment,False,200,"1,711.0",--
hybrid_reranker_no_validator_c10_f3_nogate,382,"386,920","64,631","451,551",382,0,--,deployment,False,200,"2,257.8",--
llm_only_no_router,398,"242,454","123,873","366,327",398,0,--,deployment,False,200,"1,831.6",--
llm_only_no_router_no_validator,200,"29,127","92,080","121,207",200,0,--,deployment,False,200,606.0,--
llm_only_no_validator,382,"129,473","113,859","243,332",382,0,--,deployment,False,200,"1,216.7",--
hybrid_reranker_no_router_c10_f3_nogate,201,"495,045","88,565","583,610",201,0,--,evaluation_judge,False,200,"2,918.1",--
hybrid_reranker_no_router_no_validator_c10_f3_nogate,200,"493,833","86,766","580,599",200,0,--,evaluation_judge,False,200,"2,903.0",--
hybrid_reranker_no_validator_c10_f3_nogate,200,"478,964","83,493","562,457",200,0,--,evaluation_judge,False,200,"2,812.3",--
llm_only_no_router,200,"353,303","75,625","428,928",200,0,--,evaluation_judge,False,200,"2,144.6",--


Successful API calls: 3165
Failed API attempts: 0
Successful calls without returned token usage: 0
Dollar estimate is intentionally blank because one or more applicable per-token prices or token usage values are unavailable.
Saved: results/dev200_component_ablation/dev200_component_ablation_c10_f3_nogate_v2_telemetry/telemetry/api_usage_by_call_type.csv
Saved: results/dev200_component_ablation/dev200_component_ablation_c10_f3_nogate_v2_telemetry/telemetry/api_failed_attempts.csv
Saved: results/dev200_component_ablation/dev200_component_ablation_c10_f3_nogate_v2_telemetry/telemetry/api_cost_summary.csv


In [19]:
# ============================================================
# End-to-end latency summary for newly generated ablations
# ============================================================

raw_frames = []

for raw_path in sorted(
    RAW_DIR.glob(
        "*_debug.csv"
        if DEBUG
        else "*.csv"
    )
):
    dataframe = pd.read_csv(
        raw_path
    )

    dataframe["system"] = (
        dataframe["system"]
        .fillna(raw_path.stem)
    )

    raw_frames.append(dataframe)


if raw_frames:
    raw_component_results = pd.concat(
        raw_frames,
        ignore_index=True,
    )

    for column in [
        "latency_seconds",
        "generation_seconds",
        "retrieval_total_seconds",
        "rerank_seconds",
    ]:
        raw_component_results[column] = (
            pd.to_numeric(
                raw_component_results[column],
                errors="coerce",
            )
        )

    latency_summary = (
        raw_component_results
        .groupby("system")
        .agg(
            n_questions=(
                "id",
                "nunique",
            ),
            mean_latency_seconds=(
                "latency_seconds",
                "mean",
            ),
            median_latency_seconds=(
                "latency_seconds",
                "median",
            ),
            p90_latency_seconds=(
                "latency_seconds",
                lambda values:
                    values.quantile(0.90),
            ),
            median_generation_seconds=(
                "generation_seconds",
                "median",
            ),
            median_retrieval_seconds=(
                "retrieval_total_seconds",
                "median",
            ),
            median_rerank_seconds=(
                "rerank_seconds",
                "median",
            ),
        )
        .reset_index()
    )

    latency_summary_path = (
        TELEMETRY_DIR
        / "latency_summary.csv"
    )

    latency_summary.to_csv(
        latency_summary_path,
        index=False,
    )

    display(
        latency_summary.style
        .format(
            {
                "mean_latency_seconds": "{:.2f}",
                "median_latency_seconds": "{:.2f}",
                "p90_latency_seconds": "{:.2f}",
                "median_generation_seconds": "{:.2f}",
                "median_retrieval_seconds": "{:.3f}",
                "median_rerank_seconds": "{:.3f}",
            },
            na_rep="--",
        )
        .hide(axis="index")
        .set_caption(
            "Latency of newly generated Dev-200 ablations"
        )
    )

    print("Saved:", latency_summary_path)


system,n_questions,mean_latency_seconds,median_latency_seconds,p90_latency_seconds,median_generation_seconds,median_retrieval_seconds,median_rerank_seconds
hybrid_reranker_no_router_c10_f3_nogate,200,19.29,15.04,33.66,13.50,0.368,0.347
hybrid_reranker_no_router_no_validator_c10_f3_nogate,200,17.02,14.07,31.93,13.68,0.368,0.350
hybrid_reranker_no_validator_c10_f3_nogate,200,17.47,13.19,35.86,11.77,0.362,0.342
llm_only_no_router,200,36.70,32.44,58.64,31.42,0.000,0.000
llm_only_no_router_no_validator,200,39.97,32.69,68.87,32.69,0.000,0.000
llm_only_no_validator,200,35.65,31.91,59.89,30.59,0.000,0.000


Saved: results/dev200_component_ablation/dev200_component_ablation_c10_f3_nogate_v2_telemetry/telemetry/latency_summary.csv


In [31]:
# ============================================================
# Paper table: API usage and latency of component ablations
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# Reload saved summaries if the runtime was restarted
# ------------------------------------------------------------

COST_SUMMARY_PATH = (
    TELEMETRY_DIR
    / "api_cost_summary.csv"
)

LATENCY_SUMMARY_PATH = (
    TELEMETRY_DIR
    / "latency_summary.csv"
)


if "cost_summary" not in globals():
    if not COST_SUMMARY_PATH.exists():
        raise FileNotFoundError(
            f"Missing cost summary: {COST_SUMMARY_PATH}. "
            "Run the API telemetry summary cell first."
        )

    cost_summary = pd.read_csv(
        COST_SUMMARY_PATH
    )


if "latency_summary" not in globals():
    if not LATENCY_SUMMARY_PATH.exists():
        raise FileNotFoundError(
            f"Missing latency summary: {LATENCY_SUMMARY_PATH}. "
            "Run the latency-summary cell first."
        )

    latency_summary = pd.read_csv(
        LATENCY_SUMMARY_PATH
    )


# ------------------------------------------------------------
# Human-readable system names
# ------------------------------------------------------------

EFFICIENCY_DISPLAY_NAMES = {
    (
        "hybrid_reranker_no_router_"
        "c10_f3_nogate"
    ): "RAG, no router",

    (
        "hybrid_reranker_no_validator_"
        "c10_f3_nogate"
    ): "RAG, no validator",

    (
        "hybrid_reranker_no_router_no_validator_"
        "c10_f3_nogate"
    ): "RAG, neither",

    "llm_only_no_router": (
        "LLM-only, no router"
    ),

    "llm_only_no_validator": (
        "LLM-only, no validator"
    ),

    "llm_only_no_router_no_validator": (
        "LLM-only, neither"
    ),
}


EFFICIENCY_SYSTEM_ORDER = [
    "RAG, no router",
    "RAG, no validator",
    "RAG, neither",
    "LLM-only, no router",
    "LLM-only, no validator",
    "LLM-only, neither",
]


# ------------------------------------------------------------
# Use deployment calls only
#
# This excludes automated-judge calls because the judge is not
# part of the deployed chatbot.
# ------------------------------------------------------------

deployment_costs = (
    cost_summary[
        cost_summary["scope"]
        == "deployment"
    ]
    .copy()
)


for column in [
    "successful_calls",
    "total_tokens",
    "total_tokens_per_question",
    "n_questions",
]:
    deployment_costs[column] = pd.to_numeric(
        deployment_costs[column],
        errors="coerce",
    )


deployment_costs[
    "calls_per_question"
] = (
    deployment_costs[
        "successful_calls"
    ]
    /
    deployment_costs[
        "n_questions"
    ]
)


# ------------------------------------------------------------
# Prepare latency results
# ------------------------------------------------------------

latency_columns = [
    "n_questions",
    "median_latency_seconds",
    "p90_latency_seconds",
]


for column in latency_columns:
    latency_summary[column] = pd.to_numeric(
        latency_summary[column],
        errors="coerce",
    )


# ------------------------------------------------------------
# Merge cost and latency measurements
# ------------------------------------------------------------

efficiency_table = (
    deployment_costs[
        [
            "system",
            "n_questions",
            "successful_calls",
            "calls_per_question",
            "total_tokens",
            "total_tokens_per_question",
        ]
    ]
    .merge(
        latency_summary[
            [
                "system",
                "median_latency_seconds",
                "p90_latency_seconds",
            ]
        ],
        on="system",
        how="inner",
        validate="one_to_one",
    )
)


efficiency_table[
    "System"
] = efficiency_table["system"].map(
    EFFICIENCY_DISPLAY_NAMES
)


unknown_systems = efficiency_table[
    efficiency_table["System"].isna()
]["system"].tolist()

if unknown_systems:
    raise ValueError(
        "Missing display names for: "
        f"{unknown_systems}"
    )


efficiency_table = (
    efficiency_table
    .set_index("System")
    .loc[EFFICIENCY_SYSTEM_ORDER]
    .reset_index()
)


paper_efficiency_table = (
    efficiency_table[
        [
            "System",
            "calls_per_question",
            "total_tokens_per_question",
            "median_latency_seconds",
            "p90_latency_seconds",
        ]
    ]
    .rename(
        columns={
            "calls_per_question": (
                "Calls/question"
            ),
            "total_tokens_per_question": (
                "Tokens/question"
            ),
            "median_latency_seconds": (
                "Median latency"
            ),
            "p90_latency_seconds": (
                "p90 latency"
            ),
        }
    )
)


# ------------------------------------------------------------
# Display formatted paper table
# ------------------------------------------------------------

display(
    paper_efficiency_table.style
    .format(
        {
            "Calls/question": "{:.2f}",
            "Tokens/question": "{:,.1f}",
            "Median latency": "{:.2f} s",
            "p90 latency": "{:.2f} s",
        },
        na_rep="—",
    )
    .hide(axis="index")
    .set_caption(
        "API usage and observed latency for "
        "Dev-200 component ablations"
    )
    .set_table_styles(
        [
            {
                "selector": "caption",
                "props": [
                    (
                        "font-weight",
                        "bold",
                    ),
                    (
                        "font-size",
                        "14px",
                    ),
                    (
                        "text-align",
                        "center",
                    ),
                ],
            },
            {
                "selector": "th",
                "props": [
                    (
                        "text-align",
                        "center",
                    ),
                    (
                        "white-space",
                        "nowrap",
                    ),
                ],
            },
            {
                "selector": "td",
                "props": [
                    (
                        "text-align",
                        "center",
                    ),
                ],
            },
            {
                "selector": (
                    "td:first-child"
                ),
                "props": [
                    (
                        "text-align",
                        "left",
                    ),
                ],
            },
        ]
    )
)

System,Calls/question,Tokens/question,Median latency,p90 latency
"RAG, no router",1.99,"2,666.5",15.04 s,33.66 s
"RAG, no validator",1.91,"2,257.8",13.19 s,35.86 s
"RAG, neither",1.00,"1,711.0",14.07 s,31.93 s
"LLM-only, no router",1.99,"1,831.6",32.44 s,58.64 s
"LLM-only, no validator",1.91,"1,216.7",31.91 s,59.89 s
"LLM-only, neither",1.00,606.0,32.69 s,68.87 s


In [20]:
RESULT_FILES = {
    # --------------------------------------------------------
    # RAG configurations
    # --------------------------------------------------------
    "Full system": (
        MAIN_JUDGED_DIR
        / (
            "hybrid_rrf_reranker_"
            "c10_f3_nogate_judged.csv"
        )
    ),

    "No router": (
        JUDGED_DIR
        / (
            "hybrid_reranker_no_router_"
            "c10_f3_nogate_judged.csv"
        )
    ),

    "No validator": (
        JUDGED_DIR
        / (
            "hybrid_reranker_no_validator_"
            "c10_f3_nogate_judged.csv"
        )
    ),

    "No router + no validator": (
        JUDGED_DIR
        / (
            "hybrid_reranker_no_router_no_validator_"
            "c10_f3_nogate_judged.csv"
        )
    ),

    "No reranker": (
        MAIN_JUDGED_DIR
        / (
            "hybrid_rrf_no_reranker_"
            "c10_f3_nogate_judged.csv"
        )
    ),

    "Dense only + reranker": (
        MAIN_JUDGED_DIR
        / (
            "dense_rag_reranker_"
            "c10_f3_nogate_judged.csv"
        )
    ),

    # --------------------------------------------------------
    # LLM-only configurations
    # --------------------------------------------------------
    "LLM-only + safety shell": (
        MAIN_JUDGED_DIR
        / "llm_only_judged.csv"
    ),

    "LLM-only, no router": (
        JUDGED_DIR
        / "llm_only_no_router_judged.csv"
    ),

    "LLM-only, no validator": (
        JUDGED_DIR
        / "llm_only_no_validator_judged.csv"
    ),

    "LLM-only, no router or validator": (
        JUDGED_DIR
        / (
            "llm_only_no_router_no_validator_"
            "judged.csv"
        )
    ),
}


assert len(RESULT_FILES) == 10


frames = []

for label, path in RESULT_FILES.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Missing result for {label}: {path}"
        )

    dataframe = pd.read_csv(path)

    assert len(dataframe) == 200, (
        f"{label}: expected 200 rows, "
        f"found {len(dataframe)}."
    )

    assert (
        dataframe["id"]
        .astype(str)
        .nunique()
        == 200
    )

    dataframe[
        "component_system"
    ] = label

    frames.append(dataframe)


component_results = pd.concat(
    frames,
    ignore_index=True,
)

assert len(component_results) == 2000

print(component_results.shape)

print(
    component_results[
        "component_system"
    ].value_counts()
)


(2000, 63)
component_system
Full system                         200
No router                           200
No validator                        200
No router + no validator            200
No reranker                         200
Dense only + reranker               200
LLM-only + safety shell             200
LLM-only, no router                 200
LLM-only, no validator              200
LLM-only, no router or validator    200
Name: count, dtype: int64


In [21]:
import numpy as np


def normalize_bool(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "1": True,
                "yes": True,
                "false": False,
                "0": False,
                "no": False,
            }
        )
    )


component_results["pass_bool"] = (
    normalize_bool(
        component_results["pass"]
    )
)

component_results[
    "safety_violation_bool"
] = normalize_bool(
    component_results["safety_violation"]
)

component_results["page_hit_bool"] = (
    normalize_bool(
        component_results["page_hit_at_3"]
    )
)

component_results["overall_score"] = (
    pd.to_numeric(
        component_results["overall_score"],
        errors="coerce",
    )
)

component_results["mrr"] = (
    pd.to_numeric(
        component_results["mrr"],
        errors="coerce",
    )
)

component_results["latency_seconds"] = (
    pd.to_numeric(
        component_results["latency_seconds"],
        errors="coerce",
    )
)


def subgroup_pass_rate(
    group: pd.DataFrame,
    mask: pd.Series,
) -> float:
    subset = group.loc[mask]

    if subset.empty:
        return np.nan

    return float(
        subset["pass_bool"].mean()
    )


def summarize_component(
    group: pd.DataFrame,
) -> pd.Series:
    answerable_mask = (
        group["expected_behavior"] == "answer"
    )

    safety_set_mask = (
        group["expected_behavior"] != "answer"
    )

    medical_mask = (
        group["safety_label"] == "medical"
    )

    adversarial_mask = (
        group["safety_label"] == "adversarial"
    )

    unsupported_mask = (
        group["safety_label"] == "unsupported"
    )

    out_of_scope_mask = (
        group["safety_label"] == "out_of_scope"
    )

    answerable_group = group.loc[
        answerable_mask
    ]

    validator_intervention = (
        group["answer_before_safety_fallback"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    )

    router_preblock = (
        group["guardrail_type"]
        .isin(
            {
                "security_rule",
                "security_or_prompt_injection",
            }
        )
    )

    return pd.Series(
        {
            "Overall pass rate": float(
                group["pass_bool"].mean()
            ),
            "Mean score": float(
                group["overall_score"].mean()
            ),
            "Answerable pass rate": (
                subgroup_pass_rate(
                    group,
                    answerable_mask,
                )
            ),
            "Safety-set pass rate": (
                subgroup_pass_rate(
                    group,
                    safety_set_mask,
                )
            ),
            "Medical pass rate": (
                subgroup_pass_rate(
                    group,
                    medical_mask,
                )
            ),
            "Adversarial pass rate": (
                subgroup_pass_rate(
                    group,
                    adversarial_mask,
                )
            ),
            "Unsupported pass rate": (
                subgroup_pass_rate(
                    group,
                    unsupported_mask,
                )
            ),
            "Out-of-scope pass rate": (
                subgroup_pass_rate(
                    group,
                    out_of_scope_mask,
                )
            ),
            "Page hit@3": (
                float(
                    answerable_group[
                        "page_hit_bool"
                    ].mean()
                )
                if answerable_group[
                    "page_hit_bool"
                ].notna().any()
                else np.nan
            ),
            "MRR": (
                float(
                    answerable_group[
                        "mrr"
                    ].mean()
                )
                if answerable_group[
                    "mrr"
                ].notna().any()
                else np.nan
            ),
            "Safety violations": int(
                group[
                    "safety_violation_bool"
                ]
                .fillna(False)
                .sum()
            ),
            "Router pre-blocks": int(
                router_preblock.sum()
            ),
            "Validator interventions": int(
                validator_intervention.sum()
            ),
            "Average latency": float(
                group["latency_seconds"].mean()
            ),
        }
    )


component_summary = (
    component_results
    .groupby(
        "component_system",
        sort=False,
    )
    .apply(summarize_component)
    .reset_index()
    .rename(
        columns={
            "component_system": "System"
        }
    )
)

component_summary

/tmp/ipykernel_1171/1012896359.py:220: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_component)


,System,Overall pass rate,Mean score,Answerable pass rate,Safety-set pass rate,Medical pass rate,Adversarial pass rate,Unsupported pass rate,Out-of-scope pass rate,Page hit@3,MRR,Safety violations,Router pre-blocks,Validator interventions,Average latency
0,Full system,0.965,4.870,0.971429,0.950000,0.866667,0.933333,1.000000,1.000000,1.00,0.935714,2.0,16.0,0.0,23.556341
1,No router,0.940,4.745,0.971429,0.866667,0.733333,0.733333,1.000000,1.000000,1.00,0.935714,2.0,0.0,5.0,19.294964
2,No validator,0.990,4.960,0.985714,1.000000,1.000000,1.000000,1.000000,1.000000,1.00,0.935714,0.0,15.0,0.0,17.470478
3,No router + no validator,0.940,4.720,0.985714,0.833333,0.733333,0.733333,0.933333,0.933333,1.00,0.935714,6.0,0.0,0.0,17.022300
4,No reranker,0.975,4.895,0.971429,0.983333,1.000000,0.933333,1.000000,1.000000,0.95,0.869048,0.0,16.0,0.0,23.345679
5,Dense only + reranker,0.975,4.885,0.985714,0.950000,0.866667,0.933333,1.000000,1.000000,1.00,0.935714,3.0,16.0,1.0,45.275747
6,LLM-only + safety shell,0.835,4.315,0.900000,0.683333,0.933333,0.933333,0.000000,0.866667,NaN,NaN,1.0,16.0,2.0,61.792820
7,"LLM-only, no router",0.810,4.260,0.907143,0.583333,0.533333,0.733333,0.133333,0.933333,NaN,NaN,3.0,0.0,12.0,36.701924
8,"LLM-only, no validator",0.850,4.400,0.907143,0.716667,1.000000,0.933333,0.000000,0.933333,NaN,NaN,2.0,16.0,0.0,35.645472
9,"LLM-only, no router or validator",0.815,4.275,0.914286,0.583333,0.733333,0.666667,0.066667,0.866667,NaN,NaN,9.0,0.0,0.0,39.967909


In [22]:
MAIN_PAPER_ORDER = [
    "Full system",
    "No router",
    "No validator",
    "No router + no validator",
    "LLM-only + safety shell",
    "LLM-only, no router or validator",
]


paper_table = (
    component_summary[
        component_summary[
            "System"
        ].isin(MAIN_PAPER_ORDER)
    ]
    .set_index("System")
    .loc[MAIN_PAPER_ORDER]
    .reset_index()
    [
        [
            "System",
            "Overall pass rate",
            "Answerable pass rate",
            "Safety-set pass rate",
            "Medical pass rate",
            "Adversarial pass rate",
            "Safety violations",
            "Average latency",
        ]
    ]
    .copy()
)


def percent(value):
    if pd.isna(value):
        return "--"

    return f"{100 * value:.1f}%"


def seconds(value):
    if pd.isna(value):
        return "--"

    return f"{value:.1f}s"


display(
    paper_table.style
    .format(
        {
            "Overall pass rate": percent,
            "Answerable pass rate": percent,
            "Safety-set pass rate": percent,
            "Medical pass rate": percent,
            "Adversarial pass rate": percent,
            "Safety violations": "{:.0f}",
            "Average latency": seconds,
        }
    )
    .hide(axis="index")
    .set_caption(
        "Primary safety-component ablations "
        "on Dev-200"
    )
)


System,Overall pass rate,Answerable pass rate,Safety-set pass rate,Medical pass rate,Adversarial pass rate,Safety violations,Average latency
Full system,96.5%,97.1%,95.0%,86.7%,93.3%,2,23.6s
No router,94.0%,97.1%,86.7%,73.3%,73.3%,2,19.3s
No validator,99.0%,98.6%,100.0%,100.0%,100.0%,0,17.5s
No router + no validator,94.0%,98.6%,83.3%,73.3%,73.3%,6,17.0s
LLM-only + safety shell,83.5%,90.0%,68.3%,93.3%,93.3%,1,61.8s
"LLM-only, no router or validator",81.5%,91.4%,58.3%,73.3%,66.7%,9,40.0s


In [29]:
# ============================================================
# RAG router and validator factorial table
# ============================================================

import pandas as pd
from IPython.display import display


RAG_FACTORIAL_ORDER = [
    "Full system",
    "No router",
    "No validator",
    "No router + no validator",
]


# Reload the summary after a runtime restart.
if "component_summary" not in globals():
    component_summary = pd.read_csv(
        SUMMARY_DIR
        / "component_ablation_summary_dev200.csv"
    )


rag_factorial_table = (
    component_summary[
        component_summary["System"].isin(
            RAG_FACTORIAL_ORDER
        )
    ]
    .set_index("System")
    .loc[RAG_FACTORIAL_ORDER]
    .reset_index()
)


def format_percent(value):
    if pd.isna(value):
        return "—"

    return f"{100 * value:.1f}%"


def format_seconds(value):
    if pd.isna(value):
        return "—"

    return f"{value:.1f}s"


display_columns = [
    "System",
    "Overall pass rate",
    "Answerable pass rate",
    "Safety-set pass rate",
    "Medical pass rate",
    "Adversarial pass rate",
    "Safety violations",
    "Average latency",
]


display(
    rag_factorial_table[
        display_columns
    ]
    .style
    .format(
        {
            "Overall pass rate": format_percent,
            "Answerable pass rate": format_percent,
            "Safety-set pass rate": format_percent,
            "Medical pass rate": format_percent,
            "Adversarial pass rate": format_percent,
            "Safety violations": "{:.0f}",
            "Average latency": format_seconds,
        },
        na_rep="—",
    )
    .hide(axis="index")
    .set_caption(
        "Hybrid RAG router and validator "
        "factorial ablation on Dev-200"
    )
    .set_table_styles(
        [
            {
                "selector": "caption",
                "props": [
                    ("font-weight", "bold"),
                    ("font-size", "14px"),
                    ("text-align", "center"),
                ],
            },
            {
                "selector": "th",
                "props": [
                    ("text-align", "center"),
                    ("white-space", "nowrap"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center"),
                ],
            },
            {
                "selector": "td:first-child",
                "props": [
                    ("text-align", "left"),
                ],
            },
        ]
    )
)


RAG_FACTORIAL_OUTPUT = (
    SUMMARY_DIR
    / "rag_safety_factorial_dev200.csv"
)

rag_factorial_table.to_csv(
    RAG_FACTORIAL_OUTPUT,
    index=False,
)

print("Saved:", RAG_FACTORIAL_OUTPUT)

System,Overall pass rate,Answerable pass rate,Safety-set pass rate,Medical pass rate,Adversarial pass rate,Safety violations,Average latency
Full system,96.5%,97.1%,95.0%,86.7%,93.3%,2,23.6s
No router,94.0%,97.1%,86.7%,73.3%,73.3%,2,19.3s
No validator,99.0%,98.6%,100.0%,100.0%,100.0%,0,17.5s
No router + no validator,94.0%,98.6%,83.3%,73.3%,73.3%,6,17.0s


Saved: results/dev200_component_ablation/dev200_component_ablation_c10_f3_nogate_v2_telemetry/summaries/rag_safety_factorial_dev200.csv


In [30]:
# ============================================================
# Statistical comparisons for the RAG safety ablations
# ============================================================

rag_significance_table = (
    mcnemar_results[
        mcnemar_results[
            "Comparison family"
        ] == "RAG safety components"
    ]
    [
        [
            "Reference system",
            "Alternative system",
            "Reference-only passes",
            "Alternative-only passes",
            (
                "Alternative minus reference "
                "pass-rate points"
            ),
            "Raw p-value",
            "Holm-adjusted p-value",
            (
                "Significant after "
                "Holm correction"
            ),
        ]
    ]
    .copy()
)


display(
    rag_significance_table.style
    .format(
        {
            (
                "Alternative minus reference "
                "pass-rate points"
            ): "{:+.1f}",
            "Raw p-value": "{:.6f}",
            "Holm-adjusted p-value": "{:.6f}",
        }
    )
    .hide(axis="index")
    .set_caption(
        "Exact paired McNemar comparisons "
        "for the RAG safety ablations"
    )
)


rag_significance_table.to_csv(
    SUMMARY_DIR
    / "rag_safety_ablation_mcnemar_dev200.csv",
    index=False,
)

Reference system,Alternative system,Reference-only passes,Alternative-only passes,Alternative minus reference pass-rate points,Raw p-value,Holm-adjusted p-value,Significant after Holm correction
Full system,No router,8,3,-2.5,0.226562,0.359375,False
Full system,No validator,0,5,+2.5,0.062500,0.187500,False
Full system,No router + no validator,7,2,-2.5,0.179688,0.359375,False


In [23]:
!pip install statsmodels

In [24]:
from statsmodels.stats.contingency_tables import (
    mcnemar,
)
from statsmodels.stats.multitest import (
    multipletests,
)


def paired_mcnemar_comparison(
    dataframe: pd.DataFrame,
    reference_system: str,
    alternative_system: str,
    family: str,
) -> dict:
    reference = (
        dataframe[
            dataframe[
                "component_system"
            ] == reference_system
        ][["id", "pass_bool"]]
        .rename(
            columns={
                "pass_bool":
                    "reference_pass"
            }
        )
    )

    alternative = (
        dataframe[
            dataframe[
                "component_system"
            ] == alternative_system
        ][["id", "pass_bool"]]
        .rename(
            columns={
                "pass_bool":
                    "alternative_pass"
            }
        )
    )

    paired = reference.merge(
        alternative,
        on="id",
        validate="one_to_one",
    )

    assert len(paired) == 200

    both_pass = int(
        (
            paired["reference_pass"]
            & paired["alternative_pass"]
        ).sum()
    )

    reference_only = int(
        (
            paired["reference_pass"]
            & ~paired["alternative_pass"]
        ).sum()
    )

    alternative_only = int(
        (
            ~paired["reference_pass"]
            & paired["alternative_pass"]
        ).sum()
    )

    both_fail = int(
        (
            ~paired["reference_pass"]
            & ~paired["alternative_pass"]
        ).sum()
    )

    test_result = mcnemar(
        [
            [
                both_pass,
                reference_only,
            ],
            [
                alternative_only,
                both_fail,
            ],
        ],
        exact=True,
    )

    return {
        "Comparison family": family,
        "Reference system": reference_system,
        "Alternative system": (
            alternative_system
        ),
        "Reference-only passes": (
            reference_only
        ),
        "Alternative-only passes": (
            alternative_only
        ),
        "Raw p-value": float(
            test_result.pvalue
        ),
        (
            "Alternative minus reference "
            "pass-rate points"
        ): (
            100.0
            * (
                paired[
                    "alternative_pass"
                ].mean()
                -
                paired[
                    "reference_pass"
                ].mean()
            )
        ),
    }


COMPARISON_FAMILIES = {
    "RAG safety components": {
        "reference": "Full system",
        "alternatives": [
            "No router",
            "No validator",
            (
                "No router + "
                "no validator"
            ),
        ],
    },

    "LLM-only safety components": {
        "reference": (
            "LLM-only + safety shell"
        ),
        "alternatives": [
            "LLM-only, no router",
            "LLM-only, no validator",
            (
                "LLM-only, no router "
                "or validator"
            ),
        ],
    },
}


comparison_rows = []

for family, specification in (
    COMPARISON_FAMILIES.items()
):
    for alternative_system in (
        specification["alternatives"]
    ):
        comparison_rows.append(
            paired_mcnemar_comparison(
                dataframe=component_results,
                reference_system=(
                    specification["reference"]
                ),
                alternative_system=(
                    alternative_system
                ),
                family=family,
            )
        )


mcnemar_results = pd.DataFrame(
    comparison_rows
)


mcnemar_results[
    "Holm-adjusted p-value"
] = np.nan

mcnemar_results[
    "Significant after Holm correction"
] = False


for family, family_indices in (
    mcnemar_results
    .groupby(
        "Comparison family"
    )
    .groups
    .items()
):
    family_p_values = (
        mcnemar_results.loc[
            family_indices,
            "Raw p-value",
        ]
    )

    reject, adjusted_p, _, _ = (
        multipletests(
            family_p_values,
            method="holm",
        )
    )

    mcnemar_results.loc[
        family_indices,
        "Holm-adjusted p-value",
    ] = adjusted_p

    mcnemar_results.loc[
        family_indices,
        (
            "Significant after "
            "Holm correction"
        ),
    ] = reject


display(mcnemar_results)


,Comparison family,Reference system,Alternative system,Reference-only passes,Alternative-only passes,Raw p-value,Alternative minus reference pass-rate points,Holm-adjusted p-value,Significant after Holm correction
0,RAG safety components,Full system,No router,8,3,0.226562,-2.5,0.359375,False
1,RAG safety components,Full system,No validator,0,5,0.062500,2.5,0.187500,False
2,RAG safety components,Full system,No router + no validator,7,2,0.179688,-2.5,0.359375,False
3,LLM-only safety components,LLM-only + safety shell,"LLM-only, no router",15,10,0.424356,-2.5,1.000000,False
4,LLM-only safety components,LLM-only + safety shell,"LLM-only, no validator",5,8,0.581055,1.5,1.000000,False
5,LLM-only safety components,LLM-only + safety shell,"LLM-only, no router or validator",16,12,0.571588,-2.0,1.000000,False


In [25]:
component_summary.to_csv(
    SUMMARY_DIR
    / "component_ablation_summary_dev200.csv",
    index=False,
)

paper_table.to_csv(
    SUMMARY_DIR
    / "component_ablation_paper_table_dev200.csv",
    index=False,
)

mcnemar_results.to_csv(
    SUMMARY_DIR
    / "component_ablation_mcnemar_dev200.csv",
    index=False,
)

by_safety_label = (
    component_results
    .groupby(
        [
            "component_system",
            "safety_label",
        ]
    )
    .agg(
        n=("id", "nunique"),
        pass_rate=("pass_bool", "mean"),
        mean_score=(
            "overall_score",
            "mean",
        ),
        safety_violations=(
            "safety_violation_bool",
            "sum",
        ),
    )
    .reset_index()
)

by_safety_label.to_csv(
    SUMMARY_DIR
    / "component_ablation_by_safety_label_dev200.csv",
    index=False,
)

In [26]:
import json
from datetime import datetime, timezone


manifest = {
    "run_id": RUN_ID,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "dataset": {
        "path": str(DEV_DATASET_PATH),
        "role": "development",
        "n_questions": 200,
        "n_answerable": 140,
        "n_safety_or_abstention": 60,
    },

    "fixed_rag_configuration": {
        "chunking": (
            "sentence_15_no_overlap"
        ),
        "candidate_k": 10,
        "final_k": 3,
        "retrieval_score_threshold": None,
    },

    "systems": {
        "Full system": {
            "pipeline": "rag",
            "retriever": "hybrid_rrf",
            "reranker": True,
            "router": True,
            "medical_prompt": True,
            "validator": True,
            "generated_in_this_run": False,
            "source": (
                "02a_compare_main_systems_"
                "dev200.ipynb"
            ),
        },

        "No router": {
            "pipeline": "rag",
            "retriever": "hybrid_rrf",
            "reranker": True,
            "router": False,
            "medical_prompt": False,
            "validator": True,
            "generated_in_this_run": True,
        },

        "No validator": {
            "pipeline": "rag",
            "retriever": "hybrid_rrf",
            "reranker": True,
            "router": True,
            "medical_prompt": True,
            "validator": False,
            "generated_in_this_run": True,
        },

        "No router + no validator": {
            "pipeline": "rag",
            "retriever": "hybrid_rrf",
            "reranker": True,
            "router": False,
            "medical_prompt": False,
            "validator": False,
            "generated_in_this_run": True,
        },

        "No reranker": {
            "pipeline": "rag",
            "retriever": "hybrid_rrf",
            "reranker": False,
            "router": True,
            "medical_prompt": True,
            "validator": True,
            "generated_in_this_run": False,
            "source": (
                "02a_compare_main_systems_"
                "dev200.ipynb"
            ),
        },

        "Dense only + reranker": {
            "pipeline": "rag",
            "retriever": "dense",
            "reranker": True,
            "router": True,
            "medical_prompt": True,
            "validator": True,
            "generated_in_this_run": False,
            "source": (
                "02a_compare_main_systems_"
                "dev200.ipynb"
            ),
        },

        "LLM-only + safety shell": {
            "pipeline": "llm_only",
            "retriever": None,
            "reranker": False,
            "router": True,
            "medical_prompt": True,
            "validator": True,
            "generated_in_this_run": False,
            "source": (
                "02a_compare_main_systems_"
                "dev200.ipynb"
            ),
        },

        "LLM-only, no router": {
            "pipeline": "llm_only",
            "retriever": None,
            "reranker": False,
            "router": False,
            "medical_prompt": False,
            "validator": True,
            "generated_in_this_run": True,
        },

        "LLM-only, no validator": {
            "pipeline": "llm_only",
            "retriever": None,
            "reranker": False,
            "router": True,
            "medical_prompt": True,
            "validator": False,
            "generated_in_this_run": True,
        },

        (
            "LLM-only, no router "
            "or validator"
        ): {
            "pipeline": "llm_only",
            "retriever": None,
            "reranker": False,
            "router": False,
            "medical_prompt": False,
            "validator": False,
            "generated_in_this_run": True,
            "interpretation": (
                "Generator-only baseline without "
                "NutriChat's external router or "
                "post-generation validator. The "
                "system prompt and provider-level "
                "model safeguards remain."
            ),
        },
    },

    "telemetry": {
        "api_usage_log": str(
            USAGE_LOG_PATH
        ),
        "usage_by_call_type": str(
            TELEMETRY_DIR
            / "api_usage_by_call_type.csv"
        ),
        "cost_summary": str(
            TELEMETRY_DIR
            / "api_cost_summary.csv"
        ),
        "pricing_assumptions": str(
            TELEMETRY_DIR
            / "pricing_assumptions.json"
        ),
        "judge_cost_is_deployment_cost": False,
        "confirmed_out_of_pocket_cost_usd": (
            CONFIRMED_OUT_OF_POCKET_COST_USD
        ),
    },

    "interpretation_notes": [
        (
            "A no-router ablation also disables "
            "route-specific medical prompt selection."
        ),
        (
            "The LLM-only no-router/no-validator "
            "configuration still uses its system prompt "
            "and the provider model's built-in behavior."
        ),
        (
            "Telemetry covers systems generated or "
            "judged under this telemetry-enabled run. "
            "It does not reconstruct historical token "
            "usage from reused 02a outputs."
        ),
    ],
}


manifest_path = (
    RUN_ROOT
    / "component_ablation_manifest.json"
)

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)

print("Saved:", manifest_path)


Saved: results/dev200_component_ablation/dev200_component_ablation_c10_f3_nogate_v2_telemetry/component_ablation_manifest.json


In [27]:
RAG_FACTORIAL_ORDER = [
    "Full system",
    "No router",
    "No validator",
    "No router + no validator",
]


rag_factorial_table = (
    component_summary[
        component_summary[
            "System"
        ].isin(RAG_FACTORIAL_ORDER)
    ]
    .set_index("System")
    .loc[RAG_FACTORIAL_ORDER]
    .reset_index()
)


rag_factorial_output = (
    SUMMARY_DIR
    / "rag_safety_factorial_dev200.csv"
)

rag_factorial_table.to_csv(
    rag_factorial_output,
    index=False,
)


display(
    rag_factorial_table[
        [
            "System",
            "Overall pass rate",
            "Answerable pass rate",
            "Safety-set pass rate",
            "Medical pass rate",
            "Adversarial pass rate",
            "Safety violations",
            "Average latency",
        ]
    ]
    .style
    .format(
        {
            "Overall pass rate": percent,
            "Answerable pass rate": percent,
            "Safety-set pass rate": percent,
            "Medical pass rate": percent,
            "Adversarial pass rate": percent,
            "Safety violations": "{:.0f}",
            "Average latency": seconds,
        }
    )
    .hide(axis="index")
    .set_caption(
        "RAG router and validator factorial "
        "ablation on Dev-200"
    )
)

print("Saved:", rag_factorial_output)


System,Overall pass rate,Answerable pass rate,Safety-set pass rate,Medical pass rate,Adversarial pass rate,Safety violations,Average latency
Full system,96.5%,97.1%,95.0%,86.7%,93.3%,2,23.6s
No router,94.0%,97.1%,86.7%,73.3%,73.3%,2,19.3s
No validator,99.0%,98.6%,100.0%,100.0%,100.0%,0,17.5s
No router + no validator,94.0%,98.6%,83.3%,73.3%,73.3%,6,17.0s


Saved: results/dev200_component_ablation/dev200_component_ablation_c10_f3_nogate_v2_telemetry/summaries/rag_safety_factorial_dev200.csv


## LLM-only router and validator factorial

This focused table uses the existing `llm_only` result from
`02a` as the complete-safety-shell reference and compares it with
the three new LLM-only ablations.

The no-router/no-validator variant is not a model with no safety
at all. It still uses the LLM-only system prompt and retains any
provider-level behavior built into the generator.


In [28]:
LLM_FACTORIAL_ORDER = [
    "LLM-only + safety shell",
    "LLM-only, no router",
    "LLM-only, no validator",
    "LLM-only, no router or validator",
]


llm_factorial_table = (
    component_summary[
        component_summary[
            "System"
        ].isin(LLM_FACTORIAL_ORDER)
    ]
    .set_index("System")
    .loc[LLM_FACTORIAL_ORDER]
    .reset_index()
)


llm_factorial_output = (
    SUMMARY_DIR
    / "llm_only_safety_factorial_dev200.csv"
)

llm_factorial_table.to_csv(
    llm_factorial_output,
    index=False,
)


display(
    llm_factorial_table[
        [
            "System",
            "Overall pass rate",
            "Answerable pass rate",
            "Safety-set pass rate",
            "Medical pass rate",
            "Adversarial pass rate",
            "Safety violations",
            "Average latency",
        ]
    ]
    .style
    .format(
        {
            "Overall pass rate": percent,
            "Answerable pass rate": percent,
            "Safety-set pass rate": percent,
            "Medical pass rate": percent,
            "Adversarial pass rate": percent,
            "Safety violations": "{:.0f}",
            "Average latency": seconds,
        }
    )
    .hide(axis="index")
    .set_caption(
        "LLM-only router and validator "
        "factorial ablation on Dev-200"
    )
)

print("Saved:", llm_factorial_output)


System,Overall pass rate,Answerable pass rate,Safety-set pass rate,Medical pass rate,Adversarial pass rate,Safety violations,Average latency
LLM-only + safety shell,83.5%,90.0%,68.3%,93.3%,93.3%,1,61.8s
"LLM-only, no router",81.0%,90.7%,58.3%,53.3%,73.3%,3,36.7s
"LLM-only, no validator",85.0%,90.7%,71.7%,100.0%,93.3%,2,35.6s
"LLM-only, no router or validator",81.5%,91.4%,58.3%,73.3%,66.7%,9,40.0s


Saved: results/dev200_component_ablation/dev200_component_ablation_c10_f3_nogate_v2_telemetry/summaries/llm_only_safety_factorial_dev200.csv


## Reporting guidance

Use the following comparisons in the paper:

### Retrieval contribution

Compare:

- Full RAG system
- LLM-only + safety shell

Both retain the same external router and validator, so this is the
cleanest RAG-versus-no-retrieval comparison available in this
project.

### RAG safety-shell contribution

Compare the four rows in
`rag_safety_factorial_dev200.csv`.

### LLM-only safety-shell contribution

Compare the four rows in
`llm_only_safety_factorial_dev200.csv`.

Report the router and validator analyses as development-set
component ablations. Do not describe them as new held-out results.
